In [21]:
import typing as T
import pickle
import pathlib as P
import itertools as it

In [2]:
import numpy as np
import pandas as pd
import sklearn.preprocessing as prep
from tqdm import tqdm
import scipy.sparse as sp

In [3]:
prj_root = P.Path("__file__").absolute().parent.parent.parent
import sys
if str(prj_root) not in sys.path:
  sys.path.append(str(prj_root))

import util.metrics as hm
import sklearn.preprocessing as skp
import util.obo_parser as gp

In [4]:
ns = ["cc", "mf", "bp"]
ontology_lst = ["cellular_component", "molecular_function", "biological_process"]

In [5]:
methodpred_path = [
    (P.Path(f"/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_v3_{x}_optimized/fused_preds_union.npy"),
     P.Path(f"/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_v3_{x}_optimized/fused_labels_union.npy"),
     P.Path(f"/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_v3_{x}_optimized/common_proteins.npy"),
     P.Path(f"/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_v3_{x}_optimized/union_go_terms.npy")) for x in ns
    ]

In [6]:
def sigfunc(x):
    return 1/(1+np.exp(-x))

In [7]:
def load_pred(predpath: T.Tuple[P.Path, P.Path, P.Path, P.Path]) -> T.Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load the sprofgo prediction file.
    """
    assert isinstance(predpath, T.Tuple)
    assert len(predpath) == 4
    pred_fpath, label_fpath, prot_fpath, go_fpath = predpath
    tp_preds = np.stack([np.load(label_fpath), np.load(pred_fpath)], axis=0)
    prot_names = np.load(prot_fpath, allow_pickle=True)
    go_terms = np.load(go_fpath, allow_pickle=True)

    # mean = np.mean(preds, axis=0)
    # stds = np.std(preds, axis=0)
    # preds = sigfunc((preds - mean) / stds)
    return prot_names, go_terms, tp_preds

In [8]:
methodpreds = [load_pred(x) for x in methodpred_path]

In [9]:
[[len(tp[0]), len(tp[1]), tp[2].shape] for tp in methodpreds]

[[268, 2881, (2, 268, 2881)],
 [505, 6860, (2, 505, 6860)],
 [491, 21822, (2, 491, 21822)]]

In [10]:
nspace_path = P.Path("/data0/shaojiangyi/data/data-netgo/namespace_terms.pkl")
with open(nspace_path, "rb") as h:
   namespace_terms = pickle.load(h)

In [11]:
terms_lst = [tp[1].tolist() for tp in methodpreds]

In [12]:
[len(x) for x in terms_lst]

[2881, 6860, 21822]

In [13]:
def propagate_optimized(tp_tensor: np.ndarray | tuple[np.ndarray, np.ndarray],
                       go_rels,  # gp.Ontology
                       terms_df: pd.DataFrame,
                       min_score: float = 0.):
    """
    Optimized version of the propagate function with reduced time complexity.
    """
    targs, preds = np.copy(tp_tensor)
    assert isinstance(targs, np.ndarray) and isinstance(preds, np.ndarray)
    
    mask = preds > min_score
    preds[~mask] = 0.
    
    # Pre-compute term index mapping
    term_idx = {k: i for i, k in enumerate(terms_df.gos)}
    
    # Strategy 1: Pre-compute all ancestor mappings
    ancestor_cache = {}
    for go_id in terms_df.gos:
        supgo_set = go_rels.get_anchestors(go_id)
        ancestor_indices = [idx for t in supgo_set 
                          if (idx := term_idx.get(t)) is not None]
        ancestor_cache[go_id] = np.array(ancestor_indices, dtype=np.int32)
    
    # Process each sample
    for i in tqdm(range(preds.shape[0]), total=preds.shape[0]):
        sample_mask = mask[i]
        if not np.any(sample_mask):
            continue
            
        pred_annots = terms_df.loc[sample_mask, "gos"].tolist()
        pred_scores = preds[i, sample_mask]
        
        # Strategy 2: Batch process updates using vectorized operations
        for go_id, score in zip(pred_annots, pred_scores):
            ancestor_indices = ancestor_cache[go_id]
            
            # Update ancestors not already in mask (propagation)
            new_ancestors = ancestor_indices[~sample_mask[ancestor_indices]]
            preds[i, new_ancestors] = score
            
            # Update ancestors already in mask (take maximum)
            existing_ancestors = ancestor_indices[sample_mask[ancestor_indices]]
            preds[i, existing_ancestors] = np.maximum(preds[i, existing_ancestors], score)
    
    return targs, preds

In [14]:
# def propagate_sparse(tp_tensor: np.ndarray | T.Tuple[np.ndarray, np.ndarray],
#                     go_rels: gp.Ontology,
#                     terms_df: pd.DataFrame,
#                     min_score: float = 0.):
#     targs, preds = np.copy(tp_tensor)
    
#     mask = preds > min_score
#     preds[~mask] = 0.
    
#     # Build ancestor propagation matrix
#     term_idx = {k: i for i, k in enumerate(terms_df.gos)}
#     n_terms = len(terms_df)
    
#     # Create sparse matrix: rows=terms, cols=ancestors
#     rows, cols = [], []
#     for i, go_id in enumerate(terms_df.gos):
#         ancestors = go_rels.get_anchestors(go_id)
#         for anc in ancestors:
#             if anc in term_idx:
#                 rows.append(i)
#                 cols.append(term_idx[anc])
    
#     # Propagation matrix: if term i has ancestor j, then prop_matrix[i,j] = 1
#     prop_matrix = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), 
#                                shape=(n_terms, n_terms))
    
#     # Process all rows at once
#     for i in tqdm(range(preds.shape[0]), total=preds.shape[0]):
#         if not mask[i].any():
#             continue
            
#         # Get current predictions
#         current_preds = preds[i].copy()
        
#         # Propagate: for each term with prediction, add to all ancestors
#         propagated = prop_matrix.T @ current_preds  # ancestors get max of descendants
        
#         # Take maximum with existing predictions
#         preds[i] = np.maximum(preds[i], propagated)
    
#     return targs, preds

In [15]:
targ_dfs = [
    pd.DataFrame({"gos": terms})
    for terms in terms_lst
]

In [16]:
obo_path = P.Path("/data0/shaojiangyi/data/data-netgo/go.obo")
go_rels = gp.Ontology(obo_path, with_rels=True)

In [17]:
methodtp = [propagate_optimized(tp[2], go_rels, terms_df)
            for tp, terms_df in zip(methodpreds, targ_dfs)]

  0%|          | 0/268 [00:00<?, ?it/s]

100%|██████████| 491/491 [01:19<00:00,  6.19it/s]


In [18]:
# for t, p in methodtp:
#     assert p.shape == t.shape
#     print(hm.fmax_score(t, p, need_threshold=True, auprc=True))
""" result
((53.67138317065625, 1.0), 0.525048714668755)
((31.680198882756454, 0.8), 0.25325175339063594)
((26.68211987770605, 1.0), 0.25326884192646626)
"""

' result\n((53.67138317065625, 1.0), 0.525048714668755)\n((31.680198882756454, 0.8), 0.25325175339063594)\n((26.68211987770605, 1.0), 0.25326884192646626)\n'

In [19]:
for t, p in methodtp:
    assert p.shape == t.shape
    print(hm.f1_max(t, p, need_threshold=True, auprc=True))

((np.float64(0.6543088096829403), 0.35), np.float64(0.6961055172902281))
((np.float64(0.6394655836922836), 0.24), np.float64(0.6385418933249208))
((np.float64(0.4227887225417367), 0.25), np.float64(0.38442709129024155))


In [20]:
# original no propagation
for tp in methodpreds:
    t, p = tp[2]
    (f, t), a = hm.f1_max(t, p, need_threshold=True, auprc=True)
    print(f"({f}, {t}), {a}")

(0.6525095049789427, 0.34), 0.6858838950190722
(0.6390389551085126, 0.23), 0.6280896756968606
(0.4226472503592966, 0.25), 0.3821373909978415


In [24]:
# saving
# saving_dir = P.Path("/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_optimized")
saving_dir = P.Path("/data0/shaojiangyi/pprogo-flg-1/results/fused_three_models_esm2_v3_optimized")
saving_path = [saving_dir / f"{x}_result_prop_pred.npy" for i, x in enumerate(ns)]
for path, tp in zip(saving_path, methodtp):
    np.save(path, np.stack(tp, axis=0).astype(np.float32))

In [25]:
saving_path = [saving_dir / f"{x}_result_pred.npy" for i, x in enumerate(ns)]
for path, tp in zip(saving_path, methodpreds):
    np.save(path, tp[2].astype(np.float32))